In [ ]:
pip install folium pandas geopandas --break-system-packages

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import numpy as np
import glob
import os

#poss = "./poss_2026.json"

#df = pd.read_json(poss)

files = glob.glob("places/output/*.json")

dataframes = {
    os.path.basename(f).replace(".json", "").replace("output_", ""): pd.read_json(f)
    for f in files
}

#display(dataframes)

{'places_bruteforce_uniform':                                                center  best_score  \
 0   {'type': 'Point', 'coordinates': [13.158195699...          24   
 1   {'type': 'Point', 'coordinates': [13.193053914...          18   
 2   {'type': 'Point', 'coordinates': [13.159279135...          16   
 3   {'type': 'Point', 'coordinates': [13.158991308...           5   
 4   {'type': 'Point', 'coordinates': [13.165287190...           7   
 5   {'type': 'Point', 'coordinates': [13.164333027...           5   
 6   {'type': 'Point', 'coordinates': [13.158996528...           5   
 7   {'type': 'Point', 'coordinates': [13.182832989...           4   
 8   {'type': 'Point', 'coordinates': [13.085387968...           2   
 9   {'type': 'Point', 'coordinates': [13.150625202...           3   
 10  {'type': 'Point', 'coordinates': [13.171979145...           3   
 11  {'type': 'Point', 'coordinates': [13.311241688...           3   
 12  {'type': 'Point', 'coordinates': [13.145195783...       

In [27]:
import geopandas as gpd
from shapely.geometry import Polygon

def remove_overlapping_coverage(df, overlap_threshold=0.5):
    """
    Remove rows where coverage_area overlaps >= overlap_threshold.
    Keeps the row with the highest best_score.
    """
    #display(df)
    df = df.copy()

    # Filter out rows without valid coverage
    df = df[df["coverage_area"].notnull()]
    df = df[df["coverage_area"].apply(lambda x: len(x) > 0)]

    # Convert coverage_area → shapely polygons
    df["geometry"] = df["coverage_area"].apply(lambda x: Polygon(x[0]) if x and len(x[0]) > 0 else None)

    # Filter out rows that could not be converted to a polygon
    df = df[df["geometry"].notnull()]

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:4326")

    # Reproject so area calculations are correct (Sweden projection)
    gdf = gdf.to_crs(3006)

    # Highest score processed first
    gdf = gdf.sort_values("best_score", ascending=False).reset_index(drop=True)

    kept_indices = []

    for i, row in gdf.iterrows():
        poly = row.geometry
        keep = True

        for j in kept_indices:
            existing = gdf.loc[j].geometry

            intersection = poly.intersection(existing).area
            overlap_ratio = intersection / min(poly.area, existing.area)

            if overlap_ratio >= overlap_threshold:
                keep = False
                break

        if keep:
            kept_indices.append(i)

    result = gdf.loc[kept_indices].copy()

    # remove geometry column so dataframe matches original
    result = result.drop(columns="geometry")

    return result

In [29]:
import folium
import matplotlib
import matplotlib.colors as colors

threshold = 0.50

cities = {
    "1": "Burlöv",
    "2": "Centrum",
    "4": "Trelleborg",
    "6": "Lund",
    "7": "Åvalla",
    "9": "Eslöv-2023",
    "10": "Eslöv-2024",
    "11": "Eslöv-2025",
    "12": "Kävlinge",
}


# Use a discrete color palette
palette = matplotlib.colormaps['tab20'].colors
num_colors = len(palette)

for name, df_temp in dataframes.items():
    #display(df_temp)
    #break
    if df_temp.empty:
        continue  
    # add rank and sort it
    df_map = df_temp.copy()
    df_map = df_map[df_map["seen_crimes"] > 1]
    
    df_map = remove_overlapping_coverage(df_map, overlap_threshold=threshold)
    df_map["longitude"] = df_map["center"].apply(lambda x: x["coordinates"][0])
    df_map["latitude"] = df_map["center"].apply(lambda x: x["coordinates"][1])
    df_map.sort_values(by=["seen_crimes"], ascending=False, inplace=True)
    df_map.reset_index(drop=True, inplace=True)
    #df_map["rank"] = list(range(1, df_map.shape[0]+1))
    df_map["rank"] = df_map["seen_crimes"].rank(method="min", ascending=False).astype(int)
    
    # Determine map center
    lats = [row["center"]["coordinates"][1] for _, row in df_map.iterrows()]
    lons = [row["center"]["coordinates"][0] for _, row in df_map.iterrows()]
    map_center = [sum(lats)/len(lats), sum(lons)/len(lons)]

    
    m = folium.Map(location=map_center, tiles="CartoDB positron")
    
    # Sort by rank
    #df_sorted = df_map.sort_values('rank').reset_index(drop=True)

    #city_code = name.split("_")[0]  # assuming name starts with the digit
    city_title = "Trelleborg" #cities.get(city_code, city_code)
    title_html = f'''
        <h3 align="center" style="font-size:20px"><b>{city_title}</b></h3>
    '''
    m.get_root().html.add_child(folium.Element(title_html))
    
    for i, row in df_map.iterrows():
        
        center = row["center"]
        lon, lat = center["coordinates"]
        coverage = row["coverage_area"]
        polygon_coords = [(p[1], p[0]) for p in coverage[0]]
        
        color = colors.to_hex(palette[i % num_colors])
        
        # Polygon with tooltip
        folium.Polygon(
            locations=polygon_coords,
            color=color,
            weight=3,
            fill=True,
            fill_color=color,
            fill_opacity=0.5,
            tooltip=f"Rank: {row['rank']}<br>Unsafe places: {row['seen_crimes']}"
        ).add_to(m)
        
        # Center marker
        folium.CircleMarker(
            location=[lat, lon],
            radius=5,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=1,
            popup=f"Center: {row['rank']}"
        ).add_to(m)
        
        
        
        # Rank label
        folium.Marker(
            location=[lat, lon],
            icon=folium.DivIcon(
                html=f'''
                <div style="
                    font-size:18px;
                    font-weight:bold;
                    color:white;
                    text-shadow: 2px 2px 4px black;
                ">{row["rank"]}</div>
                '''
            )
        ).add_to(m)

        
    all_coords = []

    for _, row in df_map.iterrows():
        # center point
        lon, lat = row["center"]["coordinates"]
        all_coords.append((lat, lon))
    
        # polygon points
        if row["coverage_area"] and len(row["coverage_area"]) > 0:
            polygon = row["coverage_area"][0]
            for p in polygon:
                all_coords.append((p[1], p[0]))

    m.fit_bounds(all_coords)
    # Save HTML file
    filename = f"{name}_{city_title}_{threshold*100}-percent_map.html"
    m.save(f"places/output/html/{filename}")
    
    
    df_map[["rank", "latitude", "longitude", "seen_crimes"]].to_csv(f"places/output/csv/{name}_{threshold*100}_percent.csv", index=False)
